# vehicle3: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('33_vehicle3_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,53.283,41.630,70.192,30.177,95.584,62.881,64.897,85.658,47.456
smote,80.367,66.030,53.226,87.503,74.051,80.777,63.364,87.127,73.198
bsmote,80.124,65.645,52.793,87.010,73.896,80.453,63.673,87.090,72.885
ros,80.007,65.555,52.671,87.486,73.497,80.492,63.897,87.269,72.781
adasyn,80.636,66.326,53.072,89.374,73.103,81.239,63.549,87.436,73.481
casb,79.707,65.541,53.335,86.102,74.211,80.157,64.127,87.371,72.624
casb_relaxed,79.224,64.838,52.463,85.620,73.581,79.600,64.273,87.200,72.031
casb_edited,78.495,63.638,49.763,88.904,69.558,79.231,58.974,85.462,71.066
casb_ros,80.551,66.565,54.276,87.054,74.921,80.987,64.569,87.470,73.558


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,57.977,45.545,61.665,36.822,91.634,64.228,60.611,82.822,51.761
smote,75.529,60.933,51.402,75.465,75.941,75.703,55.605,82.475,68.231
bsmote,74.976,60.044,49.638,76.661,73.652,75.157,51.216,81.245,67.510
ros,73.001,58.439,50.306,70.991,75.544,73.268,54.508,81.815,65.720
adasyn,74.495,59.485,49.088,76.661,73.023,74.842,52.452,82.019,66.990
casb,74.478,59.841,50.901,73.605,75.782,74.693,54.619,81.990,67.160
casb_relaxed,75.209,60.421,50.325,76.417,74.443,75.430,55.030,82.314,67.815
casb_edited,74.890,59.884,47.343,83.040,68.293,75.667,50.534,81.056,67.387
casb_ros,73.632,59.047,50.915,71.285,76.570,73.927,54.987,82.021,66.339


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,60.175,48.181,62.352,39.596,91.712,65.654,66.586,86.362,54.178
smote,70.858,58.069,57.277,59.419,84.847,72.133,64.877,86.395,64.464
bsmote,72.525,59.587,56.999,63.400,83.585,73.492,63.871,86.060,66.056
ros,69.947,57.360,58.082,57.287,85.954,71.621,65.570,86.255,63.654
adasyn,72.247,59.166,56.348,62.935,83.350,73.142,63.793,86.141,65.706
casb,71.260,58.392,57.064,60.338,84.453,72.395,65.190,86.398,64.826
casb_relaxed,70.718,57.756,56.631,59.668,84.294,71.981,66.226,86.814,64.237
casb_edited,78.592,64.143,52.466,83.018,74.681,78.849,57.811,84.585,71.367
casb_ros,70.276,57.947,59.149,57.525,86.188,71.856,66.148,86.790,64.112


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,60.525,48.503,61.648,40.089,91.641,65.865,60.377,85.040,54.514
smote,76.222,62.475,54.878,73.627,79.576,76.601,60.561,85.471,69.349
bsmote,76.460,62.550,54.181,75.017,78.475,76.746,59.497,85.039,69.505
ros,75.964,61.874,53.428,74.546,77.997,76.272,59.610,85.150,68.919
adasyn,76.622,62.434,53.180,76.174,77.366,76.770,59.940,85.210,69.528
casb,76.842,62.822,53.955,76.456,77.996,77.226,60.169,85.458,69.832
casb_relaxed,77.552,63.389,53.460,78.538,76.894,77.716,60.442,85.353,70.470
casb_edited,78.380,63.607,50.731,86.545,71.605,79.075,57.435,84.397,70.994
casb_ros,77.687,63.510,53.490,78.787,76.972,77.880,59.809,85.454,70.598


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,57.990,45.965,63.964,36.671,92.643,64.657,63.118,84.970,51.977
smote,75.744,61.877,54.196,74.003,78.604,76.303,61.102,85.367,68.810
bsmote,76.021,61.957,53.403,75.522,77.402,76.462,59.564,84.859,68.989
ros,74.730,60.807,53.622,72.578,78.248,75.413,60.896,85.122,67.768
adasyn,76.000,61.853,52.922,76.286,76.711,76.498,59.933,85.201,68.926
casb,75.572,61.649,53.814,74.125,78.111,76.118,61.026,85.304,68.610
casb_relaxed,75.676,61.601,53.220,75.061,77.303,76.182,61.493,85.420,68.638
casb_edited,77.589,62.818,50.076,85.377,71.034,78.205,56.188,83.875,70.204
casb_ros,75.537,61.767,54.457,73.663,78.663,76.163,61.378,85.434,68.652


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,80.551,adasyn,80.636,-0.085,LOSS
1,f1,casb_ros,66.565,adasyn,66.326,0.239,WIN
2,precision,casb_ros,54.276,smote,53.226,1.049,WIN
3,recall,casb_edited,88.904,adasyn,89.374,-0.471,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,75.209,smote,75.529,-0.320,LOSS
1,f1,casb_relaxed,60.421,smote,60.933,-0.511,LOSS
2,precision,casb_ros,50.915,smote,51.402,-0.487,LOSS
3,recall,casb_edited,83.040,bsmote,76.661,6.379,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,78.592,bsmote,72.525,6.066,WIN
1,f1,casb_edited,64.143,bsmote,59.587,4.555,WIN
2,precision,casb_ros,59.149,ros,58.082,1.066,WIN
3,recall,casb_edited,83.018,bsmote,63.400,19.618,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,77.589,bsmote,76.021,1.568,WIN
1,f1,casb_edited,62.818,bsmote,61.957,0.861,WIN
2,precision,casb_ros,54.457,smote,54.196,0.262,WIN
3,recall,casb_edited,85.377,adasyn,76.286,9.091,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.453879,0.333333,0.818182,0.209302,0.984252,0.596777,0.688529,0.858817,0.393606,None
1,42,1,svm,smote,0.75,0.777358,0.640777,0.550000,0.767442,0.787402,0.777422,0.609632,0.864494,0.709067,None
2,42,1,svm,bsmote,1.00,0.813050,0.672566,0.542857,0.883721,0.748031,0.815876,0.589477,0.865775,0.742808,None
3,42,1,svm,ros,1.00,0.806493,0.666667,0.544118,0.860465,0.755906,0.808185,0.637925,0.877312,0.736580,None
4,42,1,svm,adasyn,0.50,0.788119,0.666667,0.603774,0.744186,0.834646,0.789416,0.627802,0.876396,0.727393,None
5,42,1,svm,casb,1.00,0.800567,0.666667,0.564516,0.813953,0.787402,0.800678,0.613301,0.874199,0.733617,None
6,42,1,svm,casb_relaxed,1.00,0.810682,0.672727,0.552239,0.860465,0.763780,0.812122,0.614610,0.874748,0.741705,None
7,42,1,svm,casb_edited,1.00,0.800110,0.655172,0.520548,0.883721,0.724409,0.804065,0.566071,0.857169,0.727641,None
8,42,1,svm,casb_ros,1.00,0.800567,0.666667,0.564516,0.813953,0.787402,0.800678,0.634726,0.874565,0.733617,None
9,42,1,knn,original,0.50,0.561540,0.459016,0.777778,0.325581,0.968504,0.647043,0.621532,0.828237,0.510278,None
